# Exercise P2.2: Nested JSON to Tabular Data
### STAT 540 — Week 2


## Overview

In this exercise, you will work with nested JSON data from a public API, flatten it into a tabular DataFrame, and practice handling the messy reality of web data. This is a preview of skills you will use extensively in Week 3.

## Task 1: Fetch JSON from a Public API

Query the GitHub API for repository information:

In [1]:
import requests
import pandas as pd

# Fetch public repositories for a GitHub user
response = requests.get("https://api.github.com/users/torvalds/repos?per_page=10")
print(f"Status code: {response.status_code}")

repos = response.json()
print(f"Number of repos returned: {len(repos)}")
print(f"Type: {type(repos)}")

# Examine the structure of one repo
import json
print(json.dumps(repos[0], indent=2)[:500])  # First 500 chars, formatted

Status code: 200
Number of repos returned: 10
Type: <class 'list'>
{
  "id": 940929652,
  "node_id": "R_kgDOOBVydA",
  "name": "1590A",
  "full_name": "torvalds/1590A",
  "private": false,
  "owner": {
    "login": "torvalds",
    "id": 1024025,
    "node_id": "MDQ6VXNlcjEwMjQwMjU=",
    "avatar_url": "https://avatars.githubusercontent.com/u/1024025?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/torvalds",
    "html_url": "https://github.com/torvalds",
    "followers_url": "https://api.github.com/users/torvalds/followers",
    "following_


**Your turn:** How is this JSON structured? Is it a flat list of objects, or are there nested fields?

> The JSON has nested fields, such as the `owner` field which contains detailed information about it.

## Task 2: Flatten with `pd.json_normalize`

In [2]:
import pandas as pd

# Basic normalization
df = pd.json_normalize(repos)
print(f"Shape: {df.shape}")
print(f"\nAll columns:\n{df.columns.tolist()}")

Shape: (10, 104)

All columns:
['id', 'node_id', 'name', 'full_name', 'private', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discussions', 'forks_count', 'mirror_url', 'archived', 'disab

**Your turn:** How many columns were created? Find at least three columns that came from nested objects (they will have dots in their names, like `owner.login`).

1. `owner.node_id`
2. `license.key`
3. `owner.url`

## Task 3: Select and Clean Useful Columns

Not all 80+ columns are useful. Select the ones that matter:

In [3]:
# Select relevant columns
df_clean = df[[
    "name", "full_name", "description",
    "stargazers_count", "forks_count", "language",
    "created_at", "updated_at",
    "owner.login", "owner.type"
]].copy()

# Rename for clarity
df_clean = df_clean.rename(columns={
    "stargazers_count": "stars",
    "forks_count": "forks",
    "owner.login": "owner",
    "owner.type": "owner_type"
})

# Parse dates
df_clean["created_at"] = pd.to_datetime(df_clean["created_at"])
df_clean["updated_at"] = pd.to_datetime(df_clean["updated_at"])

print(df_clean.dtypes)
df_clean.head()

name                        object
full_name                   object
description                 object
stars                        int64
forks                        int64
language                    object
created_at     datetime64[ns, UTC]
updated_at     datetime64[ns, UTC]
owner                       object
owner_type                  object
dtype: object


,name,full_name,description,stars,forks,language,created_at,updated_at,owner,owner_type
0,1590A,torvalds/1590A,Random odd guitar pedal design in kicad,569,21,OpenSCAD,2025-03-01 04:36:29+00:00,2026-09-08 20:47:18+00:00,torvalds,User
1,AudioNoise,torvalds/AudioNoise,Random digital audio effects,4487,217,C,2026-01-09 02:33:29+00:00,2026-09-08 20:47:23+00:00,torvalds,User
2,GuitarPedal,torvalds/GuitarPedal,Linus learns analog circuits,2294,107,C,2025-09-17 01:01:29+00:00,2026-09-08 20:47:15+00:00,torvalds,User
3,HunspellColorize,torvalds/HunspellColorize,Wrapper around 'less' to colorize spelling mis...,374,20,C,2026-01-18 19:57:03+00:00,2026-09-08 16:33:43+00:00,torvalds,User
4,libdc-for-dirk,torvalds/libdc-for-dirk,"Only use for syncing with Dirk, don't use for ...",402,51,C,2017-01-17 00:25:49+00:00,2026-09-07 16:44:06+00:00,torvalds,User


## Task 4: Analyze the Cleaned Data

In [4]:
# Basic analysis
print(f"Most starred repo: {df_clean.loc[df_clean['stars'].idxmax(), 'name']}")
print(f"Total stars: {df_clean['stars'].sum()}")
print(f"Languages used: {df_clean['language'].dropna().unique()}")

# Sort by stars
df_clean.sort_values("stars", ascending=False)[["name", "stars", "forks", "language"]]

Most starred repo: linux
Total stars: 257501
Languages used: ['OpenSCAD' 'C' 'C++']


,name,stars,forks,language
6,linux,247576,64440,C
1,AudioNoise,4487,217,C
2,GuitarPedal,2294,107,C
7,pesconvert,574,75,C
0,1590A,569,21,OpenSCAD
9,subsurface-for-dirk,470,68,C++
4,libdc-for-dirk,402,51,C
5,libgit2,384,30,C
3,HunspellColorize,374,20,C
8,ScrollWheel,371,12,C


**Your turn:** What is the most-starred repository? What language is it written in?

> The most-starred repository is the linux one, written in C.

## Task 5: Try a Different API

Choose a different public API (no authentication required) and repeat the process. Some options:

- **Open Trivia DB:** `https://opentdb.com/api.php?amount=10`
- **Universities:** `http://universities.hipolabs.com/search?country=United+States&name=fayetteville`
- **Random Users:** `https://randomuser.me/api/?results=10`
- **Dog API:** `https://dog.ceo/api/breeds/list/all`

In [7]:
import requests, pandas as pd

# Your chosen API
url = "http://universities.hipolabs.com/search?country=United+States&name=fayetteville"  # Paste your URL here
response = requests.get(url)
data = response.json()

# Explore the structure
print(type(data))
print(json.dumps(data, indent=2)[:500])

# Normalize to DataFrame
df2 = pd.json_normalize(data)   # Adjust based on structure
df2.head()

<class 'list'>
[
  {
    "alpha_two_code": "US",
    "name": "University of Arkansas - Fayetteville",
    "web_pages": [
      "http://www.uark.edu/"
    ],
    "state-province": null,
    "country": "United States",
    "domains": [
      "uark.edu"
    ]
  },
  {
    "alpha_two_code": "US",
    "name": "Fayetteville State University",
    "web_pages": [
      "http://www.uncfsu.edu/"
    ],
    "state-province": null,
    "country": "United States",
    "domains": [
      "uncfsu.edu"
    ]
  },
  {
    "alp


,alpha_two_code,name,web_pages,state-province,country,domains
0,US,University of Arkansas - Fayetteville,[http://www.uark.edu/],None,United States,[uark.edu]
1,US,Fayetteville State University,[http://www.uncfsu.edu/],None,United States,[uncfsu.edu]
2,US,Fayetteville Technical Community College,[http://www.faytechcc.edu/],None,United States,[faytechcc.edu]


**Your turn:** What API did you choose? Was the JSON flat or nested? What did you have to do to get it into a DataFrame?

> I chose the Universities API. The JSON was nested, with entries containing their own observations of variables. I just called the `json_normalize` function to convert it to a dataframe.

## Task 6: Reflection

Write 100–150 words comparing reading a clean CSV versus flattening nested JSON from an API. Address: which was easier, what additional steps JSON required, and when you would choose one format over the other.

> Reading a clean CSV is undeniably easier than working with nested JSON from an API. A CSV is inherently flat and tabular, meaning a single `pd.read_csv()` command instantly creates a usable DataFrame. In contrast, working with JSON requires several additional steps: making an HTTP GET request, parsing the raw response, and using `pd.json_normalize()` to flatten nested dictionaries. Even after flattening, JSON APIs often produce bloated DataFrames with dozens of obscure, dot-notated columns (like owner.login) that require extensive filtering, renaming, and data type conversions to become analysis-ready.

I would choose a CSV when dealing with static, pre-processed tabular data, such as historical datasets or internal spreadsheets, where the focus is immediate analysis. However, I would choose JSON when building applications that rely on live, hierarchical data from web APIs—like pulling real-time user profiles, GitHub repositories, or complex metadata where a flat CSV simply cannot capture the original multi-level structure efficiently

## Submission Checklist

- [x] GitHub API data fetched and flattened
- [x] Nested columns identified
- [x] Data cleaned, renamed, and dates parsed
- [x] Basic analysis performed
- [x] Second API explored
- [x] Reflection written
- [x] Files committed to GitHub